## From here, we evaluate our model.

In [1]:
from datasets import load_dataset

In [2]:
# Define the direct URL to the raw JSON file on the Hugging Face hub
data_url = "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/alpaca_eval.json"

In [4]:
# Load the dataset using the ’json’ builder, completely bypassing the disabled script
dataset = load_dataset("json", data_files=data_url)

# Filter the dataset to only include the ’helpful_base’ subset
helpful_base_dataset = dataset['train'].filter(lambda x: x['dataset'] =='helpful_base')

alpaca_eval.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/805 [00:00<?, ? examples/s]

In [5]:
helpful_base_dataset

Dataset({
    features: ['dataset', 'instruction', 'output', 'generator'],
    num_rows: 129
})

In [7]:
# sample 15
helpful_base_15 = helpful_base_dataset.shuffle(seed=42).select(range(15))

helpful_base_15[0].keys(), helpful_base_15[0]

(dict_keys(['dataset', 'instruction', 'output', 'generator']),
 {'dataset': 'helpful_base',
  'instruction': 'What are some good browser alternatives to Chrome?',
  'output': 'Firefox, Safari, Microsoft Edge, Opera, Vivaldi, Brave.',
  'generator': 'text_davinci_003'})

In [8]:
helpful_base_15

Dataset({
    features: ['dataset', 'instruction', 'output', 'generator'],
    num_rows: 15
})

# Loading base and pretrained model

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

base_id = "gpt2"  # or your actual base model id
dpo_id  = "cass-afk/a5"  # <- your HF model

tokenizer = AutoTokenizer.from_pretrained(base_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(base_id).to("cuda").eval()
dpo_model  = AutoModelForCausalLM.from_pretrained(dpo_id).to("cuda").eval()

@torch.no_grad()
def generate(model, instruction, max_new_tokens=200):
    inputs = tokenizer(instruction, return_tensors="pt", truncation=True).to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

rows = []
for i, ex in enumerate(helpful_base_15):
    instr = ex["instruction"]
    a_base = generate(base_model, instr)
    a_dpo  = generate(dpo_model, instr)
    rows.append({"id": i+1, "instruction": instr, "base_answer": a_base, "dpo_answer": a_dpo})

rows[0]["instruction"][:120]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/962 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/153 [00:00<?, ?B/s]

'What are some good browser alternatives to Chrome?'

In [10]:
JUDGE_PROMPT = """You are a highly qualified and impartial judge evaluating two AI models. Your task is to
determine which model provides a better, more accurate, and more helpful response to the
user’s instruction.

User Instruction: {instruction}
Model A (Base Model): {base_answer}
Model B (DPO Model): {dpo_answer}

Evaluate both models. Output ONLY your final verdict as exactly one of the following
options, with no extra text or explanation: "Model A", "Model B", or "Tie".
"""

In [42]:
import os

import os
os.environ["GEMINI_API_KEY"] = "GEMINI API KEY" #enter your gemini api key


In [45]:
from google import genai

client = genai.Client()  # reads GEMINI_API_KEY automatically

def judge_verdict_gemini(instruction, base_answer, dpo_answer, model="gemini-2.5-flash"):
    prompt = JUDGE_PROMPT.format(
        instruction=instruction,
        base_answer=base_answer,
        dpo_answer=dpo_answer,
    )

    resp = client.models.generate_content(
        model=model,
        contents=prompt,
    )

    text = (resp.text or "").strip()
    if text not in ["Model A", "Model B", "Tie"]:
        return "Tie"
    return text

In [46]:
import time

results = []

for r in rows:
    verdict = judge_verdict_gemini(
        r["instruction"],
        r["base_answer"],
        r["dpo_answer"]
    )

    results.append({
        "Sample ID": r["id"],
        "Instruction": r["instruction"][:60],
        "Winner": verdict
    })

    time.sleep(12)   # 5 requests/minute → ~12 seconds between calls

In [47]:
results

[{'Sample ID': 1,
  'Instruction': 'What are some good browser alternatives to Chrome?',
  'Winner': 'Model A'},
 {'Sample ID': 2,
  'Instruction': 'Hi, my sister and her girlfriends want me to play kickball w',
  'Winner': 'Model A'},
 {'Sample ID': 3,
  'Instruction': 'Hi, I have some falafel, but no tahini to put on them. Can y',
  'Winner': 'Tie'},
 {'Sample ID': 4,
  'Instruction': 'Can you tell me how to make chocolate chip cookies?',
  'Winner': 'Tie'},
 {'Sample ID': 5,
  'Instruction': 'How can I make bubble solution?',
  'Winner': 'Tie'},
 {'Sample ID': 6,
  'Instruction': 'How is oil turned into gasoline?',
  'Winner': 'Tie'},
 {'Sample ID': 7,
  'Instruction': 'How do I wrap a present neatly?',
  'Winner': 'Tie'},
 {'Sample ID': 8,
  'Instruction': 'What is some cool music from the 1920s?',
  'Winner': 'Model A'},
 {'Sample ID': 9,
  'Instruction': "Hi, I'd like to play ice hockey. Can you explain how the gam",
  'Winner': 'Model B'},
 {'Sample ID': 10,
  'Instruction': 'Is

In [54]:
wins_b = sum(1 for x in results if x["Winner"] == "Model B")
ties   = sum(1 for x in results if x["Winner"] == "Tie")
total  = len(results)

win_rate = ((wins_b + 0.5 * ties) / total) * 100
win_rate

40.0

In [56]:
ties

8